# 02 Table 1: Baseline Characteristics

## Objectives
- Load the combined sepsis dataset created in 01 (MIMIC-IV + eICU)
- Create **Table 1** (main): 6-column layout — Overall (Survived, Died), Derivation/MIMIC-IV (Survived, Died), External validation/eICU (Survived, Died)
- Create **Supplementary Table**: 3-column layout — Overall, MIMIC-IV, eICU
- Format for journal publication with footnotes
- Export both as Word documents to `outputs/outputs_for_manuscript/`

In [15]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])


0

In [16]:
import pandas as pd
import numpy as np
from tableone import TableOne
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Combined Data

In [17]:
# Load the combined long format data from 01
df_long = pd.read_pickle('../outputs/outputs_data/combined_sepsis_long.pkl')
print(f'Loaded combined data: {df_long.shape}')
print(f'Sources: {df_long["source"].value_counts().to_dict()}')
print(f'Unique patients: {df_long["stay_id"].nunique()}')

Loaded combined data: (10987038, 22)
Sources: {'mimic': 5513491, 'eicu': 5473547}
Unique patients: 61621


## 2. Convert to Wide Format (Patient-Level)

In [18]:
def long_to_wide_for_table1(df):
    """Convert long format to wide format for Table 1.
    Uses the value closest to ICU admission (offset=0) for each variable.
    """
    id_col = 'stay_id'
    
    # Patient-level columns
    patient_cols = ['stay_id', 'age', 'sex', 'race_ethnicity', 'vent', 'uop', 'admit', 'arf',
                    'comorbidity', 'apache3_score', 'hospital_expire_flag', 'source']
    patient_cols = [c for c in patient_cols if c in df.columns]
    patient_data = df.groupby(id_col)[patient_cols].first().reset_index(drop=True)
    
    # Pivot: select value closest to admission (min abs offset)
    df_obs = df.dropna(subset=['variable_name', 'value']).copy()
    df_obs['abs_offset'] = df_obs['observationoffset'].abs()
    idx = df_obs.groupby([id_col, 'variable_name'])['abs_offset'].idxmin()
    closest = df_obs.loc[idx]
    
    wide_vals = closest.pivot_table(
        index=id_col, columns='variable_name', values='value', aggfunc='first'
    ).reset_index()
    
    # Merge
    result = pd.merge(patient_data, wide_vals, on=id_col, how='left')
    return result

df_wide = long_to_wide_for_table1(df_long)
print(f'Wide format shape: {df_wide.shape}')
print(f'Patients by source: {df_wide["source"].value_counts().to_dict()}')

Wide format shape: (61621, 32)
Patients by source: {'eicu': 31403, 'mimic': 30218}


## 3. Data Preprocessing

Convert boolean/numeric variables to readable labels for Table 1:

In [19]:
# Convert boolean/numeric flags to readable labels (matching 04 logic)
df_wide['hospital_expire_flag'] = df_wide['hospital_expire_flag'].map({0: 'Survived', 1: 'Died'})
df_wide['vent'] = df_wide['vent'].map({False: 'No', True: 'Yes'})
df_wide['arf'] = df_wide['arf'].map({False: 'No', True: 'Yes'})

# Fix race_ethnicity: merge None values into Other/Unknown
df_wide['race_ethnicity'] = df_wide['race_ethnicity'].replace({None: 'Other/Unknown'})
df_wide.loc[df_wide['race_ethnicity'].isna(), 'race_ethnicity'] = 'Other/Unknown'
print(f'Race/Ethnicity values after merging None into Other/Unknown:')
print(df_wide['race_ethnicity'].value_counts())

# Create individual binary dummy variables for each comorbidity
# Original 'comorbidity' column has one value per patient (e.g. 'hepatic_failure', 'aids', or 'None'/NaN)
comorbidity_types = {
    'aids': 'comorbidity_aids',
    'cirrhosis': 'comorbidity_cirrhosis',
    'hepatic_failure': 'comorbidity_hepatic_failure',
    'immunosuppress': 'comorbidity_immunosuppress',
    'leukemia': 'comorbidity_leukemia',
    'lymphoma': 'comorbidity_lymphoma',
    'cancer_mets': 'comorbidity_cancer_mets',
    'mult_myeloma': 'comorbidity_mult_myeloma',
}
for raw_val, col_name in comorbidity_types.items():
    df_wide[col_name] = (df_wide['comorbidity'] == raw_val).map({True: 'Yes', False: 'No'})

# Verify
print('\nHospital mortality:')
print(df_wide['hospital_expire_flag'].value_counts())
print('\nMechanical ventilation:')
print(df_wide['vent'].value_counts())
print('\nComorbidity dummies (Yes counts):')
for raw_val, col_name in comorbidity_types.items():
    n_yes = (df_wide[col_name] == 'Yes').sum()
    print(f'  {col_name}: {n_yes}')

Race/Ethnicity values after merging None into Other/Unknown:
race_ethnicity
White            45576
Other/Unknown     7432
Black             5642
Hispanic          1683
Asian             1288
Name: count, dtype: int64

Hospital mortality:
hospital_expire_flag
Survived    52326
Died         9295
Name: count, dtype: int64

Mechanical ventilation:
vent
No     37147
Yes    24474
Name: count, dtype: int64

Comorbidity dummies (Yes counts):
  comorbidity_aids: 246
  comorbidity_cirrhosis: 1879
  comorbidity_hepatic_failure: 2977
  comorbidity_immunosuppress: 827
  comorbidity_leukemia: 807
  comorbidity_lymphoma: 755
  comorbidity_cancer_mets: 2668
  comorbidity_mult_myeloma: 67


## 4. Create Table 1

### Strategy
- **Table 1 (main)**: 6 data columns — Overall (Survived/Died), MIMIC-IV (Survived/Died), eICU (Survived/Died)
  - Create 3 separate TableOne objects (Overall, MIMIC, eICU), each grouped by `hospital_expire_flag`
- **Supplementary Table**: 3 data columns — Overall, MIMIC-IV, eICU
  - Create 1 TableOne grouped by `source`

In [20]:
# === Shared configuration ===
columns = [
    # Demographics
    'age', 'sex', 'race_ethnicity',
    # Clinical conditions & treatments
    'vent', 'arf',
    # Individual comorbidities (binary dummies)
    'comorbidity_aids', 'comorbidity_cirrhosis', 'comorbidity_hepatic_failure',
    'comorbidity_immunosuppress', 'comorbidity_leukemia', 'comorbidity_lymphoma',
    'comorbidity_cancer_mets', 'comorbidity_mult_myeloma',
    # Admission type
    'admit',
    # Severity score
    'apache3_score',
    # Vital signs
    'hr', 'map', 'temp', 'rr',
    # GCS
    'gcs_eye', 'gcs_verbal', 'gcs_motor',
    # Laboratory tests
    'wbc', 'hct', 'sodium', 'glucose', 'bun', 'scr', 'bili', 'albumin',
    # Blood gas
    'ph', 'pao2', 'pco2', 'fio2', 'aa_grad',
    # Other
    'uop',
]

categorical = [
    'sex', 'race_ethnicity', 'vent', 'arf',
    'comorbidity_aids', 'comorbidity_cirrhosis', 'comorbidity_hepatic_failure',
    'comorbidity_immunosuppress', 'comorbidity_leukemia', 'comorbidity_lymphoma',
    'comorbidity_cancer_mets', 'comorbidity_mult_myeloma',
    'admit',
]

nonnormal = ['apache3_score', 'wbc', 'bili', 'scr', 'bun', 'glucose']

# Race/Ethnicity display order
race_order = ['White', 'Black', 'Hispanic', 'Asian', 'Other/Unknown']

rename_map = {
    'age': 'Age, years',
    'sex': 'Sex',
    'race_ethnicity': 'Race/Ethnicity',
    'vent': 'Mechanical ventilation',
    'arf': 'Acute renal failure',
    'comorbidity_aids': 'AIDS',
    'comorbidity_cirrhosis': 'Cirrhosis',
    'comorbidity_hepatic_failure': 'Hepatic failure',
    'comorbidity_immunosuppress': 'Immunosuppression',
    'comorbidity_leukemia': 'Leukemia',
    'comorbidity_lymphoma': 'Lymphoma',
    'comorbidity_cancer_mets': 'Metastatic cancer',
    'comorbidity_mult_myeloma': 'Multiple myeloma',
    'admit': 'Admission type',
    'apache3_score': 'APACHE III score',
    'hr': 'Heart rate, bpm',
    'map': 'Mean arterial pressure, mmHg',
    'temp': 'Temperature, \u00b0C',
    'rr': 'Respiratory rate, /min',
    'gcs_eye': 'GCS Eye',
    'gcs_verbal': 'GCS Verbal',
    'gcs_motor': 'GCS Motor',
    'wbc': 'White blood cells, \u00d710\u00b3/\u03bcL',
    'hct': 'Hematocrit, %',
    'sodium': 'Sodium, mEq/L',
    'glucose': 'Glucose, mg/dL',
    'bun': 'Blood urea nitrogen, mg/dL',
    'scr': 'Serum creatinine, mg/dL',
    'bili': 'Bilirubin, mg/dL',
    'albumin': 'Albumin, g/dL',
    'ph': 'pH',
    'pao2': 'PaO\u2082, mmHg',
    'pco2': 'PaCO\u2082, mmHg',
    'fio2': 'FiO\u2082, %',
    'aa_grad': 'A-a gradient',
    'uop': 'Urine output, mL',
}

# === Table 1 (main): 3 TableOne objects, each grouped by mortality ===
df_mimic = df_wide[df_wide['source'] == 'mimic'].copy()
df_eicu = df_wide[df_wide['source'] == 'eicu'].copy()

t1_kwargs = dict(columns=columns, categorical=categorical, nonnormal=nonnormal,
                 groupby='hospital_expire_flag', pval=False, missing=True,
                 rename=rename_map, overall=False,
                 order={'race_ethnicity': race_order})

t1_overall = TableOne(df_wide, **t1_kwargs)
t1_mimic = TableOne(df_mimic, **t1_kwargs)
t1_eicu = TableOne(df_eicu, **t1_kwargs)

print('Table 1 \u2014 3 TableOne objects created (Overall / MIMIC / eICU, each by mortality).')

# === Supplementary Table: grouped by source ===
df_wide['cohort'] = df_wide['source'].map({'mimic': 'Derivation', 'eicu': 'External validation'})
t1_supp = TableOne(df_wide, columns=columns, categorical=categorical, nonnormal=nonnormal,
                   groupby='cohort', pval=False, missing=True, rename=rename_map,
                   overall=True, order={'cohort': ['Derivation', 'External validation'],
                                        'race_ethnicity': race_order})

print('Supplementary Table created (Overall / Derivation / External validation).')

Table 1 — 3 TableOne objects created (Overall / MIMIC / eICU, each by mortality).
Supplementary Table created (Overall / Derivation / External validation).


In [21]:
# Inspect Table 1 components
print('=== Overall (by mortality) ===')
display(t1_overall.tableone)
print('\n=== MIMIC-IV (by mortality) ===')
display(t1_mimic.tableone)
print('\n=== eICU (by mortality) ===')
display(t1_eicu.tableone)

=== Overall (by mortality) ===


Grouped by hospital_expire_flag  \
                                                                                 Missing   
n                                                                                          
Age, years, mean (SD)                                                                  0   
Sex, n (%)                                 F                                               
                                           M                                               
                                           None                                            
Race/Ethnicity, n (%)                      White                                           
                                           Black                                           
                                           Hispanic                                        
                                           Asian                                           
                                           Other/Unknown                                   
Mechanical ventilation, n (%)              No                                              
                                           Yes                                             
Acute renal failure, n (%)                 No                                              
                                           Yes                                             
AIDS, n (%)                                No                                              
                                           Yes                                             
Cirrhosis, n (%)                           No                                              
                                           Yes                                             
Hepatic failure, n (%)                     No                                              
                                           Yes                                             
Immunosuppression, n (%)                   No                                              
                                           Yes                                             
Leukemia, n (%)                            No                                              
                                           Yes                                             
Lymphoma, n (%)                            No                                              
                                           Yes                                             
Metastatic cancer, n (%)                   No                                              
                                           Yes                                             
Multiple myeloma, n (%)                    No                                              
                                           Yes                                             
Admission type, n (%)                      elective                                        
                                           emergency                                       
APACHE III score, median [Q1,Q3]                                                       0   
Heart rate, bpm, mean (SD)                                                           665   
Mean arterial pressure, mmHg, mean (SD)                                              608   
Temperature, °C, mean (SD)                                                          1873   
Respiratory rate, /min, mean (SD)                                                   1437   
GCS Eye, mean (SD)                                                                  9296   
GCS Verbal, mean (SD)                                                              16344   
GCS Motor, mean (SD)                                                                9305   
White blood cells, ×10³/μL, median [Q1,Q3]                                           707   
Hematocrit, %, mean (SD)                                                             603   
Sodium, m


=== MIMIC-IV (by mortality) ===


Grouped by hospital_expire_flag  \
                                                                                 Missing   
n                                                                                          
Age, years, mean (SD)                                                                  0   
Sex, n (%)                                 F                                               
                                           M                                               
Race/Ethnicity, n (%)                      White                                           
                                           Black                                           
                                           Hispanic                                        
                                           Asian                                           
                                           Other/Unknown                                   
Mechanical ventilation, n (%)              No                                              
                                           Yes                                             
Acute renal failure, n (%)                 No                                              
                                           Yes                                             
AIDS, n (%)                                No                                              
                                           Yes                                             
Cirrhosis, n (%)                           No                                              
                                           Yes                                             
Hepatic failure, n (%)                     No                                              
                                           Yes                                             
Immunosuppression, n (%)                   No                                              
                                           Yes                                             
Leukemia, n (%)                            No                                              
                                           Yes                                             
Lymphoma, n (%)                            No                                              
                                           Yes                                             
Metastatic cancer, n (%)                   No                                              
                                           Yes                                             
Multiple myeloma, n (%)                    No                                              
                                           Yes                                             
Admission type, n (%)                      elective                                        
                                           emergency                                       
APACHE III score, median [Q1,Q3]                                                       0   
Heart rate, bpm, mean (SD)                                                            30   
Mean arterial pressure, mmHg, mean (SD)                                               31   
Temperature, °C, mean (SD)                                                          1442   
Respiratory rate, /min, mean (SD)                                                     47   
GCS Eye, mean (SD)                                                                    40   
GCS Verbal, mean (SD)                                                               6926   
GCS Motor, mean (SD)                                                                  44   
White blood cells, ×10³/μL, median [Q1,Q3]                                           174   
Hematocrit, %, mean (SD)                                                             156   
Sodium, mEq/L, mean (SD)                                                             112   
Glucose, 


=== eICU (by mortality) ===


Grouped by hospital_expire_flag  \
                                                                                 Missing   
n                                                                                          
Age, years, mean (SD)                                                                  0   
Sex, n (%)                                 F                                               
                                           M                                               
                                           None                                            
Race/Ethnicity, n (%)                      White                                           
                                           Black                                           
                                           Hispanic                                        
                                           Asian                                           
                                           Other/Unknown                                   
Mechanical ventilation, n (%)              No                                              
                                           Yes                                             
Acute renal failure, n (%)                 No                                              
                                           Yes                                             
AIDS, n (%)                                No                                              
                                           Yes                                             
Cirrhosis, n (%)                           No                                              
                                           Yes                                             
Hepatic failure, n (%)                     No                                              
                                           Yes                                             
Immunosuppression, n (%)                   No                                              
                                           Yes                                             
Leukemia, n (%)                            No                                              
                                           Yes                                             
Lymphoma, n (%)                            No                                              
                                           Yes                                             
Metastatic cancer, n (%)                   No                                              
                                           Yes                                             
Multiple myeloma, n (%)                    No                                              
Admission type, n (%)                      elective                                        
                                           emergency                                       
APACHE III score, median [Q1,Q3]                                                       0   
Heart rate, bpm, mean (SD)                                                           635   
Mean arterial pressure, mmHg, mean (SD)                                              577   
Temperature, °C, mean (SD)                                                           431   
Respiratory rate, /min, mean (SD)                                                   1390   
GCS Eye, mean (SD)                                                                  9256   
GCS Verbal, mean (SD)                                                               9418   
GCS Motor, mean (SD)                                                                9261   
White blood cells, ×10³/μL, median [Q1,Q3]                                           533   
Hematocrit, %, mean (SD)                                                             447   
Sodium, mEq/L, mean (SD)                                                             632   
Glucose, 

## 5. Export to Word Documents

### 5.1 Helper functions and row extraction

In [22]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import nsdecls
from docx.oxml import parse_xml

output_dir = '../outputs/outputs_for_manuscript'
os.makedirs(output_dir, exist_ok=True)


def set_cell_border(cell, **kwargs):
    """Set cell borders (journal-style: horizontal only)."""
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    xml = (
        '<w:tcBorders %s>'
        '<w:top w:val="%s" w:sz="%s" w:space="0" w:color="000000"/>'
        '<w:bottom w:val="%s" w:sz="%s" w:space="0" w:color="000000"/>'
        '<w:left w:val="none" w:sz="0" w:space="0" w:color="000000"/>'
        '<w:right w:val="none" w:sz="0" w:space="0" w:color="000000"/>'
        '</w:tcBorders>'
    ) % (nsdecls('w'),
         kwargs.get('top', 'none'), kwargs.get('top_sz', '4'),
         kwargs.get('bottom', 'none'), kwargs.get('bottom_sz', '4'))
    tcPr.append(parse_xml(xml))


def set_cell_text(cell, text, bold=False, italic=False, indent=False,
                  align='left', size=Pt(9)):
    """Set cell text with formatting."""
    cell.text = ''
    p = cell.paragraphs[0]
    if align == 'center':
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    elif align == 'right':
        p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    else:
        p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    if indent:
        p.paragraph_format.left_indent = Inches(0.15)
    run = p.add_run(text)
    run.font.name = 'Times New Roman'
    run.font.size = size
    run.bold = bold
    run.italic = italic
    p.paragraph_format.space_before = Pt(1)
    p.paragraph_format.space_after = Pt(1)


# ---------- Binary variable handling ----------
BINARY_VARS = {
    'Sex':                     {'keep': 'F',   'display': 'Female, n (%)'},
    'Mechanical ventilation':  {'keep': 'Yes', 'display': 'Mechanical ventilation, n (%)'},
    'Acute renal failure':     {'keep': 'Yes', 'display': 'Acute renal failure, n (%)'},
    'AIDS':                    {'keep': 'Yes', 'display': 'AIDS, n (%)'},
    'Cirrhosis':               {'keep': 'Yes', 'display': 'Cirrhosis, n (%)'},
    'Hepatic failure':         {'keep': 'Yes', 'display': 'Hepatic failure, n (%)'},
    'Immunosuppression':       {'keep': 'Yes', 'display': 'Immunosuppression, n (%)'},
    'Leukemia':                {'keep': 'Yes', 'display': 'Leukemia, n (%)'},
    'Lymphoma':                {'keep': 'Yes', 'display': 'Lymphoma, n (%)'},
    'Metastatic cancer':       {'keep': 'Yes', 'display': 'Metastatic cancer, n (%)'},
    'Multiple myeloma':        {'keep': 'Yes', 'display': 'Multiple myeloma, n (%)'},
}

_STAT_SUFFIXES = [', n (%)', ', mean (SD)', ', median [Q1,Q3]']


def _strip_stat_suffix(s):
    """Remove TableOne stat suffix from variable name."""
    for sfx in _STAT_SUFFIXES:
        if s.endswith(sfx):
            return s[:-len(sfx)]
    return s


def extract_two_group_cols(t1_obj):
    """Extract Survived and Died values per row from a TableOne object.
    Skips the 'n' row (patient count) since it is shown in the header.
    """
    t1 = t1_obj.tableone.copy()
    t1r = t1.reset_index()
    col_list = t1.columns.tolist()

    died_col = survived_col = None
    for c in col_list:
        cstr = str(c).lower()
        if 'died' in cstr:
            died_col = c
        elif 'survived' in cstr:
            survived_col = c

    # Collect raw rows
    raw_rows = []
    for _, row in t1r.iterrows():
        var_name = str(row.iloc[0]) if pd.notna(row.iloc[0]) else ''
        level = str(row.iloc[1]) if pd.notna(row.iloc[1]) else ''
        su = str(row[survived_col]) if survived_col and pd.notna(row[survived_col]) else ''
        di = str(row[died_col]) if died_col and pd.notna(row[died_col]) else ''
        if su == 'nan': su = ''
        if di == 'nan': di = ''
        raw_rows.append({'var_name': var_name, 'level': level,
                         'survived': su, 'died': di})

    # Build final rows
    rows = []
    seen_binary = set()

    for r in raw_rows:
        vn_raw = r['var_name']
        lv = r['level']
        vn_base = _strip_stat_suffix(vn_raw)

        # Skip the 'n' row (patient count already in header)
        if vn_raw == 'n':
            continue

        # Binary variable?
        if vn_base in BINARY_VARS:
            cfg = BINARY_VARS[vn_base]
            if lv == cfg['keep']:
                rows.append({
                    'label': cfg['display'],
                    'survived': r['survived'],
                    'died': r['died'],
                    'indent': False,
                })
                seen_binary.add(vn_base)
            continue

        # Non-binary: continuous or multi-category
        if lv == '' or lv == 'None':
            rows.append({
                'label': vn_raw,
                'survived': r['survived'],
                'died': r['died'],
                'indent': False,
            })
        else:
            # Multi-category variable
            if vn_raw not in seen_binary:
                if not any(x['label'] == vn_raw and x['survived'] == '' for x in rows):
                    rows.append({
                        'label': vn_raw,
                        'survived': '',
                        'died': '',
                        'indent': False,
                    })
            rows.append({
                'label': lv,
                'survived': r['survived'],
                'died': r['died'],
                'indent': True,
            })

    return rows


# Extract rows for 6-column table
rows_overall = extract_two_group_cols(t1_overall)
rows_mimic_raw = extract_two_group_cols(t1_mimic)
rows_eicu_raw = extract_two_group_cols(t1_eicu)

# Use overall as reference; align cohort rows
lookup_mi = {r['label']: r for r in rows_mimic_raw}
lookup_ei = {r['label']: r for r in rows_eicu_raw}

rows_mimic = []
rows_eicu = []
for r in rows_overall:
    lbl = r['label']
    if r['survived'] == '' and r['died'] == '':
        rows_mimic.append({'label': lbl, 'survived': '', 'died': '', 'indent': r['indent']})
        rows_eicu.append({'label': lbl, 'survived': '', 'died': '', 'indent': r['indent']})
    else:
        mi = lookup_mi.get(lbl, {'label': lbl, 'survived': '0', 'died': '0', 'indent': r['indent']})
        ei = lookup_ei.get(lbl, {'label': lbl, 'survived': '0', 'died': '0', 'indent': r['indent']})
        rows_mimic.append(mi)
        rows_eicu.append(ei)

print(f'Rows extracted: {len(rows_overall)} (aligned across cohorts)')
for r in rows_overall:
    pfx = '  ' if r['indent'] else ''
    sv = r['survived'][:25] if r['survived'] else '-'
    di = r['died'][:25] if r['died'] else '-'
    print(f"  {pfx}{r['label']}: surv={sv}  died={di}")

# Patient counts
counts = {
    'overall_surv': len(df_wide[df_wide['hospital_expire_flag'] == 'Survived']),
    'overall_died': len(df_wide[df_wide['hospital_expire_flag'] == 'Died']),
    'mimic_surv': len(df_mimic[df_mimic['hospital_expire_flag'] == 'Survived']),
    'mimic_died': len(df_mimic[df_mimic['hospital_expire_flag'] == 'Died']),
    'eicu_surv': len(df_eicu[df_eicu['hospital_expire_flag'] == 'Survived']),
    'eicu_died': len(df_eicu[df_eicu['hospital_expire_flag'] == 'Died']),
}
for k, v in counts.items():
    print(f'  {k}: {v:,}')

print('Helper functions defined and rows extracted.')

Rows extracted: 43 (aligned across cohorts)
  Age, years, mean (SD): surv=65.0 (16.0)  died=69.1 (15.0)
  Female, n (%): surv=22472 (42.9)  died=4180 (45.0)
  Race/Ethnicity, n (%): surv=-  died=-
    White: surv=39109 (74.7)  died=6467 (69.6)
    Black: surv=4787 (9.1)  died=855 (9.2)
    Hispanic: surv=1464 (2.8)  died=219 (2.4)
    Asian: surv=1088 (2.1)  died=200 (2.2)
    Other/Unknown: surv=5878 (11.2)  died=1554 (16.7)
  Mechanical ventilation, n (%): surv=20094 (38.4)  died=4380 (47.1)
  Acute renal failure, n (%): surv=5334 (10.2)  died=2450 (26.4)
  AIDS, n (%): surv=207 (0.4)  died=39 (0.4)
  Cirrhosis, n (%): surv=1405 (2.7)  died=474 (5.1)
  Hepatic failure, n (%): surv=2068 (4.0)  died=909 (9.8)
  Immunosuppression, n (%): surv=667 (1.3)  died=160 (1.7)
  Leukemia, n (%): surv=570 (1.1)  died=237 (2.5)
  Lymphoma, n (%): surv=565 (1.1)  died=190 (2.0)
  Metastatic cancer, n (%): surv=1860 (3.6)  died=808 (8.7)
  Multiple myeloma, n (%): surv=53 (0.1)  died=14 (0.2)
  Admi

In [23]:
# =====================================================================
# Table 1 (main): 7 columns = Characteristic + 6 data columns
# Characteristic | Overall Survived | Overall Died |
#                | MIMIC Survived   | MIMIC Died   |
#                | eICU Survived    | eICU Died    |
# =====================================================================

def build_footnotes():
    """Return shared footnote texts."""
    return [
        ('Abbreviations: A-a gradient, alveolar-arterial oxygen gradient; '
         'AIDS, acquired immunodeficiency syndrome; '
         'APACHE, Acute Physiology and Chronic Health Evaluation; '
         'BUN, blood urea nitrogen; '
         'eICU, eICU Collaborative Research Database; '
         'FiO\u2082, fraction of inspired oxygen; '
         'GCS, Glasgow Coma Scale; '
         'MAP, mean arterial pressure; '
         'MIMIC-IV, Medical Information Mart for Intensive Care IV; '
         'PaCO\u2082, partial pressure of arterial carbon dioxide; '
         'PaO\u2082, partial pressure of arterial oxygen; '
         'SD, standard deviation; '
         'SOFA, Sequential Organ Failure Assessment.'),
        ('Data are presented as mean (SD) for normally distributed continuous variables, '
         'median [Q1, Q3] for non-normally distributed continuous variables, '
         'and n (%) for categorical variables.'),
        ('The derivation cohort was drawn from MIMIC-IV '
         '(Beth Israel Deaconess Medical Center, 2008\u20132019). '
         'The external validation cohort was drawn from eICU Collaborative Research Database '
         '(208 hospitals across the United States, 2014\u20132015).'),
        ('All patients met Sepsis-3 criteria (suspected infection with a '
         'Sequential Organ Failure Assessment score \u2265 2).'),
        ('Urine output is the total urine output during the first 24 hours '
         'of ICU admission.'),
    ]


def add_footnotes(doc, footnote_texts):
    """Add footnotes paragraph to document."""
    fn = doc.add_paragraph()
    fn.paragraph_format.space_before = Pt(6)
    fn.paragraph_format.space_after = Pt(2)
    for fi, ft in enumerate(footnote_texts):
        if fi > 0:
            fn.add_run('\n')
        run = fn.add_run(ft)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(8)


def create_doc_landscape():
    """Create a landscape-oriented Word document."""
    doc = Document()
    section = doc.sections[0]
    # Landscape: swap width/height
    section.page_width = Inches(11)
    section.page_height = Inches(8.5)
    section.left_margin = Inches(0.7)
    section.right_margin = Inches(0.7)
    section.top_margin = Inches(0.8)
    section.bottom_margin = Inches(0.8)
    style = doc.styles['Normal']
    style.font.name = 'Times New Roman'
    style.font.size = Pt(9)
    return doc


# --- Build Table 1 (main, 6-column) ---
doc = create_doc_landscape()

# Title
tp = doc.add_paragraph()
tp.alignment = WD_ALIGN_PARAGRAPH.LEFT
run = tp.add_run('Table 1. ')
run.bold = True
run.font.name = 'Times New Roman'
run.font.size = Pt(11)
run = tp.add_run(
    'Baseline Characteristics of Patients With Sepsis '
    'Stratified by Cohort and Hospital Mortality'
)
run.font.name = 'Times New Roman'
run.font.size = Pt(11)

# Table: header row 1 (merged span) + header row 2 (sub-columns) + data rows
n_cols = 7  # Characteristic + 6 data
n_data_rows = len(rows_overall)
n_rows_total = 2 + n_data_rows  # 2 header rows + data

word_table = doc.add_table(rows=n_rows_total, cols=n_cols)
word_table.alignment = WD_TABLE_ALIGNMENT.CENTER

# --- Header row 1: group labels with merged cells ---
n_overall = counts['overall_surv'] + counts['overall_died']
n_mimic = counts['mimic_surv'] + counts['mimic_died']
n_eicu = counts['eicu_surv'] + counts['eicu_died']

# Merge cells for span headers
cell_char = word_table.cell(0, 0)
set_cell_text(cell_char, '', bold=True, size=Pt(8))
set_cell_border(cell_char, top='single', top_sz='12', bottom='single', bottom_sz='4')

span_headers = [
    (1, 2, f'Overall (N = {n_overall:,})'),
    (3, 4, f'Derivation (n = {n_mimic:,})'),
    (5, 6, f'External validation (n = {n_eicu:,})'),
]
for start, end, text in span_headers:
    merged = word_table.cell(0, start).merge(word_table.cell(0, end))
    set_cell_text(merged, text, bold=True, align='center', size=Pt(8))
    set_cell_border(merged, top='single', top_sz='12', bottom='single', bottom_sz='4')

# --- Header row 2: sub-column labels ---
sub_headers = [
    'Characteristic',
    f'Survived\n(n={counts["overall_surv"]:,})',
    f'Died\n(n={counts["overall_died"]:,})',
    f'Survived\n(n={counts["mimic_surv"]:,})',
    f'Died\n(n={counts["mimic_died"]:,})',
    f'Survived\n(n={counts["eicu_surv"]:,})',
    f'Died\n(n={counts["eicu_died"]:,})',
]
for j, sh in enumerate(sub_headers):
    cell = word_table.cell(1, j)
    set_cell_text(cell, sh, bold=True,
                  align='center' if j > 0 else 'left', size=Pt(8))
    set_cell_border(cell, bottom='single', bottom_sz='6')

# --- Data rows ---
for i in range(n_data_rows):
    row_idx = i + 2
    r_ov = rows_overall[i]
    r_mi = rows_mimic[i]
    r_ei = rows_eicu[i]

    is_header = (r_ov['survived'] == '' and r_ov['died'] == '')

    cell = word_table.cell(row_idx, 0)
    set_cell_text(cell, r_ov['label'], bold=is_header, indent=r_ov['indent'], size=Pt(8))

    data_vals = [
        r_ov['survived'], r_ov['died'],
        r_mi['survived'], r_mi['died'],
        r_ei['survived'], r_ei['died'],
    ]
    for j, val in enumerate(data_vals):
        cell = word_table.cell(row_idx, j + 1)
        set_cell_text(cell, val, align='center', size=Pt(8))

    # Borders
    if i == n_data_rows - 1:
        for j in range(n_cols):
            set_cell_border(word_table.cell(row_idx, j), bottom='single', bottom_sz='12')
    else:
        for j in range(n_cols):
            set_cell_border(word_table.cell(row_idx, j))

# Column widths
widths = [Inches(2.0)] + [Inches(1.3)] * 6
for row_obj in word_table.rows:
    for j, w in enumerate(widths):
        row_obj.cells[j].width = w

# Footnotes
add_footnotes(doc, build_footnotes())

# Save
path_table1 = os.path.join(output_dir, 'Table1.docx')
doc.save(path_table1)
print(f'Table 1 saved: {path_table1} ({os.path.getsize(path_table1)/1024:.1f} KB)')

Table 1 saved: ../outputs/outputs_for_manuscript\Table1.docx (40.6 KB)


In [24]:
# =====================================================================
# Supplementary Table: 4 columns = Characteristic + Overall + Derivation + External validation
# =====================================================================

t1s = t1_supp.tableone.copy()
t1s_reset = t1s.reset_index()
col_list_s = t1s.columns.tolist()

overall_col_s = deriv_col_s = valid_col_s = None
for c in col_list_s:
    cstr = str(c).lower()
    if 'overall' in cstr:
        overall_col_s = c
    elif 'derivation' in cstr:
        deriv_col_s = c
    elif 'external' in cstr or 'validation' in cstr:
        valid_col_s = c

# Collect raw rows
raw_rows_s = []
for _, row in t1s_reset.iterrows():
    var_name = str(row.iloc[0]) if pd.notna(row.iloc[0]) else ''
    level = str(row.iloc[1]) if pd.notna(row.iloc[1]) else ''
    ov = str(row[overall_col_s]) if overall_col_s and pd.notna(row[overall_col_s]) else ''
    dv = str(row[deriv_col_s]) if deriv_col_s and pd.notna(row[deriv_col_s]) else ''
    vv = str(row[valid_col_s]) if valid_col_s and pd.notna(row[valid_col_s]) else ''
    if ov == 'nan': ov = ''
    if dv == 'nan': dv = ''
    if vv == 'nan': vv = ''
    raw_rows_s.append({'var_name': var_name, 'level': level,
                       'overall': ov, 'derivation': dv, 'validation': vv})

# Build rows with binary variable handling
rows_supp = []
for r in raw_rows_s:
    vn_raw = r['var_name']
    lv = r['level']
    vn_base = _strip_stat_suffix(vn_raw)

    # Skip the 'n' row (patient count already in header)
    if vn_raw == 'n':
        continue

    # Binary variable?
    if vn_base in BINARY_VARS:
        cfg = BINARY_VARS[vn_base]
        if lv == cfg['keep']:
            rows_supp.append({
                'label': cfg['display'],
                'overall': r['overall'],
                'derivation': r['derivation'],
                'validation': r['validation'],
                'indent': False,
            })
        continue

    # Non-binary: continuous or multi-category
    if lv == '' or lv == 'None':
        rows_supp.append({
            'label': vn_raw,
            'overall': r['overall'],
            'derivation': r['derivation'],
            'validation': r['validation'],
            'indent': False,
        })
    else:
        # Multi-category: emit header if first time
        if not any(x['label'] == vn_raw and x['overall'] == '' for x in rows_supp):
            rows_supp.append({
                'label': vn_raw,
                'overall': '', 'derivation': '', 'validation': '',
                'indent': False,
            })
        rows_supp.append({
            'label': lv,
            'overall': r['overall'],
            'derivation': r['derivation'],
            'validation': r['validation'],
            'indent': True,
        })

print(f'Supplementary rows: {len(rows_supp)}')

# Build Word document (portrait)
doc2 = Document()
section2 = doc2.sections[0]
section2.page_width = Inches(8.5)
section2.page_height = Inches(11)
section2.left_margin = Inches(1)
section2.right_margin = Inches(1)
section2.top_margin = Inches(1)
section2.bottom_margin = Inches(1)
doc2.styles['Normal'].font.name = 'Times New Roman'
doc2.styles['Normal'].font.size = Pt(10)

# Title
tp2 = doc2.add_paragraph()
tp2.alignment = WD_ALIGN_PARAGRAPH.LEFT
run = tp2.add_run('Supplementary Table. ')
run.bold = True
run.font.name = 'Times New Roman'
run.font.size = Pt(11)
run = tp2.add_run(
    'Baseline Characteristics of Patients With Sepsis '
    'by Derivation and External Validation Cohorts'
)
run.font.name = 'Times New Roman'
run.font.size = Pt(11)

n_overall_all = len(df_wide)

# Table
nc2 = 4
nr2 = len(rows_supp) + 1
wt2 = doc2.add_table(rows=nr2, cols=nc2)
wt2.alignment = WD_TABLE_ALIGNMENT.CENTER

hdrs2 = [
    'Characteristic',
    f'Overall\n(N = {n_overall_all:,})',
    f'Derivation\n(n = {len(df_mimic):,})',
    f'External validation\n(n = {len(df_eicu):,})',
]
for j, h in enumerate(hdrs2):
    cell = wt2.cell(0, j)
    set_cell_text(cell, h, bold=True, align='center' if j > 0 else 'left', size=Pt(9))
    set_cell_border(cell, top='single', top_sz='12', bottom='single', bottom_sz='6')

for i, r in enumerate(rows_supp):
    ri = i + 1
    is_hdr = (r['overall'] == '' and r['derivation'] == '' and r['validation'] == '')
    cell = wt2.cell(ri, 0)
    set_cell_text(cell, r['label'], bold=is_hdr, indent=r['indent'], size=Pt(9))
    for j, key in enumerate(['overall', 'derivation', 'validation']):
        cell = wt2.cell(ri, j + 1)
        set_cell_text(cell, r[key], align='center', size=Pt(9))
    if i == len(rows_supp) - 1:
        for j in range(nc2):
            set_cell_border(wt2.cell(ri, j), bottom='single', bottom_sz='12')
    else:
        for j in range(nc2):
            set_cell_border(wt2.cell(ri, j))

widths2 = [Inches(2.6), Inches(1.4), Inches(1.4), Inches(1.4)]
for row_obj in wt2.rows:
    for j, w in enumerate(widths2):
        row_obj.cells[j].width = w

add_footnotes(doc2, build_footnotes())

# Save
path_supp = os.path.join(output_dir, 'Table1_supplementary.docx')
doc2.save(path_supp)
print(f'Supplementary Table saved: {path_supp} ({os.path.getsize(path_supp)/1024:.1f} KB)')
print('\nBoth tables exported successfully.')

Supplementary rows: 43
Supplementary Table saved: ../outputs/outputs_for_manuscript\Table1_supplementary.docx (39.2 KB)

Both tables exported successfully.


In [25]:
# =====================================================================
# Export Markdown tables for embedding in manuscript.md
# =====================================================================

def build_md_table_main(rows_ov, rows_mi, rows_ei, counts):
    """Build a markdown table string for the 6-column main Table 1."""
    lines = []
    lines.append(f'| Characteristic | Overall Survived (n={counts["overall_surv"]:,}) '
                 f'| Overall Died (n={counts["overall_died"]:,}) '
                 f'| Derivation Survived (n={counts["mimic_surv"]:,}) '
                 f'| Derivation Died (n={counts["mimic_died"]:,}) '
                 f'| External validation Survived (n={counts["eicu_surv"]:,}) '
                 f'| External validation Died (n={counts["eicu_died"]:,}) |')
    lines.append('|:---|:---:|:---:|:---:|:---:|:---:|:---:|')

    for i in range(len(rows_ov)):
        r_ov = rows_ov[i]
        r_mi = rows_mi[i]
        r_ei = rows_ei[i]
        label = r_ov['label']
        if r_ov['indent']:
            label = f'&ensp;{label}'
        sv_ov = r_ov['survived'] or ''
        di_ov = r_ov['died'] or ''
        sv_mi = r_mi['survived'] or ''
        di_mi = r_mi['died'] or ''
        sv_ei = r_ei['survived'] or ''
        di_ei = r_ei['died'] or ''
        # Bold header rows (no data)
        if sv_ov == '' and di_ov == '':
            label = f'**{label}**'
        lines.append(f'| {label} | {sv_ov} | {di_ov} | {sv_mi} | {di_mi} | {sv_ei} | {di_ei} |')

    return '\n'.join(lines)


def build_md_table_supp(rows_s, n_overall, n_deriv, n_valid):
    """Build a markdown table string for the supplementary table."""
    lines = []
    lines.append(f'| Characteristic | Overall (N={n_overall:,}) '
                 f'| Derivation (n={n_deriv:,}) '
                 f'| External validation (n={n_valid:,}) |')
    lines.append('|:---|:---:|:---:|:---:|')

    for r in rows_s:
        label = r['label']
        if r['indent']:
            label = f'&ensp;{label}'
        ov = r['overall'] or ''
        dv = r['derivation'] or ''
        vv = r['validation'] or ''
        if ov == '' and dv == '' and vv == '':
            label = f'**{label}**'
        lines.append(f'| {label} | {ov} | {dv} | {vv} |')

    return '\n'.join(lines)


# Generate markdown strings
md_table1 = build_md_table_main(rows_overall, rows_mimic, rows_eicu, counts)
md_supp = build_md_table_supp(rows_supp, len(df_wide), len(df_mimic), len(df_eicu))

# Save markdown files
md_table1_path = os.path.join(output_dir, 'Table1.md')
md_supp_path = os.path.join(output_dir, 'Table1_supplementary.md')

with open(md_table1_path, 'w', encoding='utf-8') as f:
    f.write('**Table 1.** Baseline Characteristics of Patients With Sepsis '
            'Stratified by Cohort and Hospital Mortality\n\n')
    f.write(md_table1)
    f.write('\n\nAbbreviations: A-a gradient, alveolar-arterial oxygen gradient; '
            'AIDS, acquired immunodeficiency syndrome; '
            'APACHE, Acute Physiology and Chronic Health Evaluation; '
            'BUN, blood urea nitrogen; '
            'FiO\u2082, fraction of inspired oxygen; '
            'GCS, Glasgow Coma Scale; '
            'MAP, mean arterial pressure; '
            'PaCO\u2082, partial pressure of arterial carbon dioxide; '
            'PaO\u2082, partial pressure of arterial oxygen; '
            'SD, standard deviation; '
            'SOFA, Sequential Organ Failure Assessment.\n')
    f.write('Data are presented as mean (SD), median [Q1, Q3], or n (%).\n')

with open(md_supp_path, 'w', encoding='utf-8') as f:
    f.write('**Supplementary Table.** Baseline Characteristics of Patients With Sepsis '
            'by Derivation and External Validation Cohorts\n\n')
    f.write(md_supp)
    f.write('\n\nAbbreviations: A-a gradient, alveolar-arterial oxygen gradient; '
            'AIDS, acquired immunodeficiency syndrome; '
            'APACHE, Acute Physiology and Chronic Health Evaluation; '
            'BUN, blood urea nitrogen; '
            'FiO\u2082, fraction of inspired oxygen; '
            'GCS, Glasgow Coma Scale; '
            'MAP, mean arterial pressure; '
            'PaCO\u2082, partial pressure of arterial carbon dioxide; '
            'PaO\u2082, partial pressure of arterial oxygen; '
            'SD, standard deviation; '
            'SOFA, Sequential Organ Failure Assessment.\n')
    f.write('Data are presented as mean (SD), median [Q1, Q3], or n (%).\n')

print(f'Markdown Table 1 saved: {md_table1_path}')
print(f'Markdown Supplementary Table saved: {md_supp_path}')

# ---- Auto-update manuscript.md ----
manuscript_path = '../manuscript.md'
with open(manuscript_path, 'r', encoding='utf-8') as f:
    manuscript = f.read()

# Build the replacement block
table_block = (
    '**Table 1.** Baseline Characteristics of Patients With Sepsis '
    'Stratified by Cohort and Hospital Mortality '
    '([Word](outputs/outputs_for_manuscript/Table1.docx))\n\n'
    + md_table1 + '\n\n'
    '**Supplementary Table.** Baseline Characteristics by Cohort '
    '([Word](outputs/outputs_for_manuscript/Table1_supplementary.docx))\n\n'
    + md_supp + '\n\n'
    '> Generated by `02_table1.ipynb`. Data source: '
    '`outputs/outputs_data/combined_sepsis_long.pkl` (MIMIC-IV + eICU, N=61,621).'
)

# Find and replace the existing table section
import re
pattern = re.compile(
    r'\*\*\[?Table 1[\s\S]*?Generated by `02_table1\.ipynb`[^\n]*',
    re.MULTILINE
)
if pattern.search(manuscript):
    manuscript_new = pattern.sub(table_block, manuscript)
    with open(manuscript_path, 'w', encoding='utf-8') as f:
        f.write(manuscript_new)
    print(f'manuscript.md updated with inline markdown tables.')
else:
    print('WARNING: Could not find Table 1 section in manuscript.md. Manual update needed.')
    print('Expected pattern not found.')

Markdown Table 1 saved: ../outputs/outputs_for_manuscript\Table1.md
Markdown Supplementary Table saved: ../outputs/outputs_for_manuscript\Table1_supplementary.md
Expected pattern not found.
